# CRUD Operations

Reusable Python functions for Create, Read, Update, and Delete operations on the `cps714.restaurant_manager` schema. This notebook is the **single source of truth** for all CRUD functions.

Checkout operations and the demo are in [Restaurant Manager Functions](#notebook-360701697965264), which `%run`s this notebook.

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

CATALOG = "cps714"
SCHEMA = "cps714.restaurant_manager"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA restaurant_manager")

def _q(value):
    """Quote a Python value for safe interpolation into SQL."""
    if value is None:
        return "NULL"
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    if isinstance(value, (int, float)):
        return str(value)
    # Escape single quotes by doubling them
    return "'" + str(value).replace("'", "''") + "'"

# Create the server_table_assignment mapping table if it doesn't already exist.
_existing = spark.sql(f"SHOW TABLES IN {SCHEMA} LIKE 'server_table_assignment'").collect()
if len(_existing) == 0:
    spark.sql(f"""
        CREATE TABLE {SCHEMA}.server_table_assignment (
            assignment_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
            table_id      BIGINT,
            employee_id   BIGINT,
            assigned_at   TIMESTAMP,
            status        STRING
        )
    """)
    print(f"Created {SCHEMA}.server_table_assignment.")
else:
    print(f"{SCHEMA}.server_table_assignment already exists — skipping creation.")

In [0]:
# ════════════════════════════════════════════════════════════
# 1. RESTAURANT TABLE CRUD
# ════════════════════════════════════════════════════════════
# create_restaurant_table, get_restaurant_table, get_all_tables,
# update_restaurant_table, delete_restaurant_table

def create_restaurant_table(restaurant_id, table_number, capacity=4):
    """
    Create a new restaurant table and its associated seats.
    
    Parameters:
        restaurant_id (int): The restaurant this table belongs to.
        table_number   (int): The human-visible table number.
        capacity       (int): Number of seats at the table (default 4).
    
    Returns:
        int: The new table_id.
    """
    # Validate restaurant exists
    restaurant = spark.sql(f"SELECT 1 FROM {SCHEMA}.restaurant WHERE restaurant_id = {restaurant_id}")
    if restaurant.count() == 0:
        raise ValueError(f"Restaurant {restaurant_id} does not exist.")
    
    # Check for duplicate table_number in the same restaurant
    existing = spark.sql(f"""
        SELECT 1 FROM {SCHEMA}.restaurant_table 
        WHERE restaurant_id = {restaurant_id} AND table_number = {table_number}
    """)
    if existing.count() > 0:
        raise ValueError(f"Table number {table_number} already exists in restaurant {restaurant_id}.")
    
    # Insert the new table row (table_id is GENERATED ALWAYS AS IDENTITY — omit it)
    spark.sql(f"""
        INSERT INTO {SCHEMA}.restaurant_table (restaurant_id, table_number, capacity, status)
        VALUES ({restaurant_id}, {table_number}, {capacity}, 'available')
    """)
    
    # Retrieve the auto-generated table_id
    new_table_id = spark.sql(f"""
        SELECT table_id FROM {SCHEMA}.restaurant_table 
        WHERE restaurant_id = {restaurant_id} AND table_number = {table_number}
        ORDER BY table_id DESC LIMIT 1
    """).collect()[0]["table_id"]
    
    # Generate seats for this table (seat_id is GENERATED ALWAYS AS IDENTITY — omit it)
    seat_values = ", ".join(f"({new_table_id}, {i})" for i in range(1, capacity + 1))
    spark.sql(f"""
        INSERT INTO {SCHEMA}.seat (table_id, seat_number)
        VALUES {seat_values}
    """)
    
    print(f"Created table {table_number} (id={new_table_id}) with {capacity} seats in restaurant {restaurant_id}.")
    return new_table_id

def get_restaurant_table(table_id):
    """
    Retrieve a single restaurant table by its ID.
    
    Parameters:
        table_id (int): The table to look up.
    
    Returns:
        Row: The table row, or None if not found.
    """
    result = spark.sql(f"""
        SELECT t.table_id, t.restaurant_id, t.table_number, t.capacity, t.status,
               r.name AS restaurant_name
        FROM {SCHEMA}.restaurant_table t
        JOIN {SCHEMA}.restaurant r ON t.restaurant_id = r.restaurant_id
        WHERE t.table_id = {table_id}
    """)
    if result.count() == 0:
        print(f"Table {table_id} not found.")
        return None
    row = result.collect()[0]
    print(f"Table {row['table_number']} (id={row['table_id']}), capacity={row['capacity']}, status={row['status']}, restaurant={row['restaurant_name']}.")
    return row

def get_all_tables(restaurant_id):
    """
    List all restaurant tables for a given restaurant.
    
    Parameters:
        restaurant_id (int): The restaurant whose tables to list.
    
    Returns:
        DataFrame: All tables for the restaurant.
    """
    df = spark.sql(f"""
        SELECT t.table_id, t.table_number, t.capacity, t.status,
               (SELECT COUNT(*) FROM {SCHEMA}.seat s WHERE s.table_id = t.table_id) AS seat_count,
               (SELECT COUNT(*) FROM {SCHEMA}.seat_order_item soi
                JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
                WHERE s.table_id = t.table_id) AS order_count
        FROM {SCHEMA}.restaurant_table t
        WHERE t.restaurant_id = {restaurant_id}
        ORDER BY t.table_number
    """)
    print(f"Tables for restaurant {restaurant_id}:")
    display(df)
    return df

def update_restaurant_table(table_id, table_number=None, capacity=None, status=None):
    """
    Update a restaurant table's number, capacity, or status.
    
    Parameters:
        table_id      (int): The table to update.
        table_number  (int, optional): New table number.
        capacity      (int, optional): New capacity.
        status        (str, optional): New status ('available' or 'occupied').
    
    Returns:
        bool: True if the table was updated.
    """
    # Validate table exists
    table = spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    if table.count() == 0:
        raise ValueError(f"Table {table_id} does not exist.")
    
    # Build SET clause dynamically
    set_clauses = []
    if table_number is not None:
        # Check for duplicate table_number
        dup = spark.sql(f"""
            SELECT 1 FROM {SCHEMA}.restaurant_table
            WHERE table_number = {table_number} AND table_id != {table_id}
        """)
        if dup.count() > 0:
            raise ValueError(f"Table number {table_number} is already in use.")
        set_clauses.append(f"table_number = {table_number}")
    if capacity is not None:
        if capacity < 1:
            raise ValueError("Capacity must be at least 1.")
        set_clauses.append(f"capacity = {capacity}")
    if status is not None:
        if status not in ('available', 'occupied'):
            raise ValueError("Status must be 'available' or 'occupied'.")
        set_clauses.append(f"status = '{status}'")
    
    if not set_clauses:
        print("No fields to update.")
        return False
    
    set_sql = ", ".join(set_clauses)
    spark.sql(f"UPDATE {SCHEMA}.restaurant_table SET {set_sql} WHERE table_id = {table_id}")
    print(f"Updated table {table_id}: {set_sql}.")
    return True

def delete_restaurant_table(table_id):
    """
    Delete a restaurant table and all its associated seats, orders, and server assignments.
    
    Parameters:
        table_id (int): The table to delete.
    
    Returns:
        bool: True if the table was deleted.
    """
    # Validate table exists
    table = spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    if table.count() == 0:
        raise ValueError(f"Table {table_id} does not exist.")
    
    table_row = table.collect()[0]
    table_number = table_row["table_number"]
    
    # 1) Delete all seat_order_items for seats at this table
    spark.sql(f"""
        DELETE FROM {SCHEMA}.seat_order_item
        WHERE seat_id IN (SELECT seat_id FROM {SCHEMA}.seat WHERE table_id = {table_id})
    """)
    
    # 2) Delete all seats for this table
    spark.sql(f"DELETE FROM {SCHEMA}.seat WHERE table_id = {table_id}")
    
    # 3) Delete all server_table_assignments for this table
    spark.sql(f"DELETE FROM {SCHEMA}.server_table_assignment WHERE table_id = {table_id}")
    
    # 4) Delete the table itself
    spark.sql(f"DELETE FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    
    print(f"Deleted table {table_number} (id={table_id}) and all associated seats, orders, and assignments.")
    return True

In [0]:
# ════════════════════════════════════════════════════════════
# 2. SERVER-TABLE ASSIGNMENT CRUD
# ════════════════════════════════════════════════════════════
# assign_table_to_server, get_table_assignment, get_server_tables,
# update_table_assignment, remove_server_from_table

def assign_table_to_server(table_id, employee_id):
    """
    Assign (or reassign) a restaurant table to a server (employee with role='server').
    
    Parameters:
        table_id    (int): The table to assign.
        employee_id (int): The server (employee) to assign it to.
    
    Returns:
        int: The assignment_id.
    """
    # Validate table exists
    table = spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    if table.count() == 0:
        raise ValueError(f"Table {table_id} does not exist.")
    
    # Validate employee exists and is a server
    emp = spark.sql(f"""
        SELECT * FROM {SCHEMA}.employee 
        WHERE employee_id = {employee_id} AND role = 'server' AND status = 'active'
    """)
    if emp.count() == 0:
        raise ValueError(f"Employee {employee_id} is not an active server.")
    
    # Check if table already has an active assignment; if so, deactivate it
    existing = spark.sql(f"""
        SELECT assignment_id FROM {SCHEMA}.server_table_assignment 
        WHERE table_id = {table_id} AND status = 'active'
    """)
    if existing.count() > 0:
        spark.sql(f"""
            UPDATE {SCHEMA}.server_table_assignment 
            SET status = 'inactive' 
            WHERE table_id = {table_id} AND status = 'active'
        """)
        print(f"Deactivated previous assignment for table {table_id}.")
    
    # Insert new assignment (assignment_id is GENERATED ALWAYS AS IDENTITY — omit it)
    spark.sql(f"""
        INSERT INTO {SCHEMA}.server_table_assignment (table_id, employee_id, assigned_at, status)
        VALUES ({table_id}, {employee_id}, current_timestamp(), 'active')
    """)
    
    # Retrieve the auto-generated assignment_id
    new_assignment_id = spark.sql(f"""
        SELECT assignment_id FROM {SCHEMA}.server_table_assignment 
        WHERE table_id = {table_id} AND status = 'active'
        ORDER BY assignment_id DESC LIMIT 1
    """).collect()[0]["assignment_id"]
    
    # Update table status to 'occupied'
    spark.sql(f"""
        UPDATE {SCHEMA}.restaurant_table 
        SET status = 'occupied' 
        WHERE table_id = {table_id}
    """)
    
    # Get server name for the message
    emp_row = emp.collect()[0]
    server_name = emp_row["first_name"] + " " + emp_row["last_name"]
    print(f"Assigned table {table_id} to server {server_name} (employee_id={employee_id}).")
    return new_assignment_id

def get_table_assignment(table_id):
    """
    Get the active server assignment for a table.
    
    Parameters:
        table_id (int): The table to check.
    
    Returns:
        Row: The active assignment row with server details, or None.
    """
    result = spark.sql(f"""
        SELECT a.assignment_id, a.table_id, t.table_number,
               e.employee_id, e.first_name, e.last_name, e.role,
               a.status, a.assigned_at
        FROM {SCHEMA}.server_table_assignment a
        JOIN {SCHEMA}.restaurant_table t ON a.table_id = t.table_id
        JOIN {SCHEMA}.employee e ON a.employee_id = e.employee_id
        WHERE a.table_id = {table_id} AND a.status = 'active'
    """)
    if result.count() == 0:
        print(f"No active server assignment for table {table_id}.")
        return None
    row = result.collect()[0]
    print(f"Table {row['table_number']} is assigned to {row['first_name']} {row['last_name']} ({row['role']}, id={row['employee_id']}).")
    return row

def get_server_tables(employee_id):
    """
    List all tables currently assigned to a server.
    
    Parameters:
        employee_id (int): The server to look up.
    
    Returns:
        DataFrame: All active table assignments for the server.
    """
    df = spark.sql(f"""
        SELECT a.assignment_id, a.table_id, t.table_number, t.capacity, t.status,
               a.assigned_at
        FROM {SCHEMA}.server_table_assignment a
        JOIN {SCHEMA}.restaurant_table t ON a.table_id = t.table_id
        WHERE a.employee_id = {employee_id} AND a.status = 'active'
        ORDER BY a.assigned_at DESC
    """)
    print(f"Active table assignments for employee {employee_id}:")
    display(df)
    return df

def update_table_assignment(table_id, new_employee_id):
    """
    Reassign a table from its current server to a new server.
    Deactivates the old active assignment and creates a new one.
    
    Parameters:
        table_id        (int): The table to reassign.
        new_employee_id (int): The new server to assign.
    
    Returns:
        int: The new assignment_id.
    """
    # Validate table exists
    table = spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    if table.count() == 0:
        raise ValueError(f"Table {table_id} does not exist.")
    
    # Validate new employee is an active server
    emp = spark.sql(f"""
        SELECT * FROM {SCHEMA}.employee
        WHERE employee_id = {new_employee_id} AND role = 'server' AND status = 'active'
    """)
    if emp.count() == 0:
        raise ValueError(f"Employee {new_employee_id} is not an active server.")
    
    # Deactivate current active assignment
    spark.sql(f"""
        UPDATE {SCHEMA}.server_table_assignment
        SET status = 'inactive'
        WHERE table_id = {table_id} AND status = 'active'
    """)
    
    # Insert new assignment
    spark.sql(f"""
        INSERT INTO {SCHEMA}.server_table_assignment (table_id, employee_id, assigned_at, status)
        VALUES ({table_id}, {new_employee_id}, current_timestamp(), 'active')
    """)
    
    new_assignment_id = spark.sql(f"""
        SELECT assignment_id FROM {SCHEMA}.server_table_assignment
        WHERE table_id = {table_id} AND status = 'active'
        ORDER BY assignment_id DESC LIMIT 1
    """).collect()[0]["assignment_id"]
    
    emp_row = emp.collect()[0]
    print(f"Reassigned table {table_id} to {emp_row['first_name']} {emp_row['last_name']} (assignment_id={new_assignment_id}).")
    return new_assignment_id

def remove_server_from_table(table_id):
    """
    Remove the active server assignment from a table (mark as 'released').
    Does not change the table status — use clear_table for full reset.
    
    Parameters:
        table_id (int): The table to remove the server from.
    
    Returns:
        bool: True if the server was removed.
    """
    active = spark.sql(f"""
        SELECT assignment_id FROM {SCHEMA}.server_table_assignment
        WHERE table_id = {table_id} AND status = 'active'
    """)
    if active.count() == 0:
        print(f"No active server assignment to remove for table {table_id}.")
        return False
    
    spark.sql(f"""
        UPDATE {SCHEMA}.server_table_assignment
        SET status = 'released'
        WHERE table_id = {table_id} AND status = 'active'
    """)
    print(f"Removed server assignment from table {table_id}.")
    return True

In [0]:
# ════════════════════════════════════════════════════════════
# 3. SEAT CRUD
# ════════════════════════════════════════════════════════════
# get_seats, add_seat, update_seat, remove_seat

def get_seats(table_id):
    """
    List all seats for a given table, ordered by seat number.
    
    Parameters:
        table_id (int): The table whose seats to list.
    
    Returns:
        DataFrame: All seats for the table.
    """
    df = spark.sql(f"""
        SELECT s.seat_id, s.table_id, s.seat_number,
               (SELECT COUNT(*) FROM {SCHEMA}.seat_order_item soi
                WHERE soi.seat_id = s.seat_id) AS order_count
        FROM {SCHEMA}.seat s
        WHERE s.table_id = {table_id}
        ORDER BY s.seat_number
    """)
    print(f"Seats for table {table_id}:")
    display(df)
    return df

def add_seat(table_id, seat_number=None):
    """
    Add a new seat to a table. If seat_number is not provided, auto-assign
    the next available number (max + 1).
    
    Parameters:
        table_id    (int): The table to add a seat to.
        seat_number (int, optional): The seat number to assign.
    
    Returns:
        int: The new seat_id.
    """
    # Validate table exists
    table = spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    if table.count() == 0:
        raise ValueError(f"Table {table_id} does not exist.")
    
    # Auto-assign seat number if not provided
    if seat_number is None:
        max_seat = spark.sql(f"""
            SELECT COALESCE(MAX(seat_number), 0) + 1 AS next_num
            FROM {SCHEMA}.seat WHERE table_id = {table_id}
        """).collect()[0]["next_num"]
        seat_number = max_seat
    else:
        # Check for duplicate seat_number
        dup = spark.sql(f"""
            SELECT 1 FROM {SCHEMA}.seat
            WHERE table_id = {table_id} AND seat_number = {seat_number}
        """)
        if dup.count() > 0:
            raise ValueError(f"Seat number {seat_number} already exists at table {table_id}.")
    
    # Insert the seat (seat_id is GENERATED ALWAYS AS IDENTITY)
    spark.sql(f"""
        INSERT INTO {SCHEMA}.seat (table_id, seat_number)
        VALUES ({table_id}, {seat_number})
    """)
    
    new_seat_id = spark.sql(f"""
        SELECT seat_id FROM {SCHEMA}.seat
        WHERE table_id = {table_id} AND seat_number = {seat_number}
        ORDER BY seat_id DESC LIMIT 1
    """).collect()[0]["seat_id"]
    
    print(f"Added seat {seat_number} (id={new_seat_id}) to table {table_id}.")
    return new_seat_id

def update_seat(seat_id, seat_number=None):
    """
    Update a seat's number.
    
    Parameters:
        seat_id      (int): The seat to update.
        seat_number  (int, optional): The new seat number.
    
    Returns:
        bool: True if the seat was updated.
    """
    seat = spark.sql(f"SELECT * FROM {SCHEMA}.seat WHERE seat_id = {seat_id}")
    if seat.count() == 0:
        raise ValueError(f"Seat {seat_id} does not exist.")
    
    if seat_number is None:
        print("No fields to update.")
        return False
    
    seat_row = seat.collect()[0]
    table_id = seat_row["table_id"]
    
    # Check for duplicate seat_number at the same table
    dup = spark.sql(f"""
        SELECT 1 FROM {SCHEMA}.seat
        WHERE table_id = {table_id} AND seat_number = {seat_number} AND seat_id != {seat_id}
    """)
    if dup.count() > 0:
        raise ValueError(f"Seat number {seat_number} already exists at table {table_id}.")
    
    spark.sql(f"""
        UPDATE {SCHEMA}.seat SET seat_number = {seat_number}
        WHERE seat_id = {seat_id}
    """)
    print(f"Updated seat {seat_id} to seat_number={seat_number}.")
    return True

def remove_seat(seat_id):
    """
    Remove a seat from a table. Fails if the seat has active orders.
    
    Parameters:
        seat_id (int): The seat to remove.
    
    Returns:
        bool: True if the seat was removed.
    """
    seat = spark.sql(f"SELECT * FROM {SCHEMA}.seat WHERE seat_id = {seat_id}")
    if seat.count() == 0:
        raise ValueError(f"Seat {seat_id} does not exist.")
    
    # Check for active orders
    orders = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {SCHEMA}.seat_order_item
        WHERE seat_id = {seat_id}
    """).collect()[0]["cnt"]
    if orders > 0:
        raise ValueError(f"Cannot remove seat {seat_id}: it has {orders} active order(s). Remove orders first.")
    
    seat_row = seat.collect()[0]
    spark.sql(f"DELETE FROM {SCHEMA}.seat WHERE seat_id = {seat_id}")
    print(f"Removed seat {seat_row['seat_number']} (id={seat_id}) from table {seat_row['table_id']}.")
    return True

In [0]:
# ════════════════════════════════════════════════════════════
# 4. MENU ITEM CRUD
# ════════════════════════════════════════════════════════════
# create_menu_item, get_menu_item, get_all_menu_items,
# update_menu_item, delete_menu_item

def create_menu_item(restaurant_id, name, description, price, category, available=True):
    """
    Create a new menu item.
    
    Parameters:
        restaurant_id (int): The restaurant this item belongs to.
        name          (str): Item name.
        description   (str): Item description.
        price         (float): Item price.
        category      (str): Category (e.g., 'Appetizer', 'Main', 'Dessert', 'Beverage').
        available      (bool): Whether the item is available (default True).
    
    Returns:
        int: The new menu_item_id.
    """
    # Validate restaurant exists
    restaurant = spark.sql(f"SELECT 1 FROM {SCHEMA}.restaurant WHERE restaurant_id = {restaurant_id}")
    if restaurant.count() == 0:
        raise ValueError(f"Restaurant {restaurant_id} does not exist.")
    
    # Check for duplicate name in the same restaurant
    dup = spark.sql(f"""
        SELECT 1 FROM {SCHEMA}.menu_item
        WHERE restaurant_id = {restaurant_id} AND LOWER(name) = LOWER('{name}')
    """)
    if dup.count() > 0:
        raise ValueError(f"Menu item '{name}' already exists in restaurant {restaurant_id}.")
    
    avail_str = 'true' if available else 'false'
    spark.sql(f"""
        INSERT INTO {SCHEMA}.menu_item (restaurant_id, name, description, price, category, available)
        VALUES ({restaurant_id}, '{name}', '{description}', {price}, '{category}', {avail_str})
    """)
    
    new_item_id = spark.sql(f"""
        SELECT menu_item_id FROM {SCHEMA}.menu_item
        WHERE restaurant_id = {restaurant_id} AND name = '{name}'
        ORDER BY menu_item_id DESC LIMIT 1
    """).collect()[0]["menu_item_id"]
    
    print(f"Created menu item '{name}' (id={new_item_id}, price={price}, category='{category}').")
    return new_item_id

def get_menu_item(menu_item_id):
    """
    Retrieve a single menu item by ID.
    
    Parameters:
        menu_item_id (int): The menu item to look up.
    
    Returns:
        Row: The menu item row, or None.
    """
    result = spark.sql(f"""
        SELECT * FROM {SCHEMA}.menu_item WHERE menu_item_id = {menu_item_id}
    """)
    if result.count() == 0:
        print(f"Menu item {menu_item_id} not found.")
        return None
    row = result.collect()[0]
    print(f"{row['name']} (id={row['menu_item_id']}), price={row['price']}, category={row['category']}, available={row['available']}.")
    return row

def get_all_menu_items(restaurant_id, available_only=False):
    """
    List all menu items for a restaurant.
    
    Parameters:
        restaurant_id  (int): The restaurant whose menu to list.
        available_only (bool): If True, show only available items.
    
    Returns:
        DataFrame: All menu items for the restaurant.
    """
    filter_clause = f"AND available = true" if available_only else ""
    df = spark.sql(f"""
        SELECT menu_item_id, name, category, price, available, description
        FROM {SCHEMA}.menu_item
        WHERE restaurant_id = {restaurant_id} {filter_clause}
        ORDER BY category, name
    """)
    print(f"Menu items for restaurant {restaurant_id}{' (available only)' if available_only else ''}:")
    display(df)
    return df

def update_menu_item(menu_item_id, name=None, description=None, price=None, category=None, available=None):
    """
    Update a menu item's fields.
    
    Parameters:
        menu_item_id (int): The item to update.
        name         (str, optional): New name.
        description  (str, optional): New description.
        price        (float, optional): New price.
        category     (str, optional): New category.
        available    (bool, optional): New availability.
    
    Returns:
        bool: True if the item was updated.
    """
    item = spark.sql(f"SELECT * FROM {SCHEMA}.menu_item WHERE menu_item_id = {menu_item_id}")
    if item.count() == 0:
        raise ValueError(f"Menu item {menu_item_id} does not exist.")
    
    set_clauses = []
    if name is not None:
        set_clauses.append(f"name = '{name}'")
    if description is not None:
        set_clauses.append(f"description = '{description}'")
    if price is not None:
        if price < 0:
            raise ValueError("Price cannot be negative.")
        set_clauses.append(f"price = {price}")
    if category is not None:
        set_clauses.append(f"category = '{category}'")
    if available is not None:
        avail_str = 'true' if available else 'false'
        set_clauses.append(f"available = {avail_str}")
    
    if not set_clauses:
        print("No fields to update.")
        return False
    
    set_sql = ", ".join(set_clauses)
    spark.sql(f"UPDATE {SCHEMA}.menu_item SET {set_sql} WHERE menu_item_id = {menu_item_id}")
    print(f"Updated menu item {menu_item_id}: {set_sql}.")
    return True

def delete_menu_item(menu_item_id):
    """
    Delete a menu item. Fails if it has active orders.
    
    Parameters:
        menu_item_id (int): The menu item to delete.
    
    Returns:
        bool: True if the item was deleted.
    """
    item = spark.sql(f"SELECT * FROM {SCHEMA}.menu_item WHERE menu_item_id = {menu_item_id}")
    if item.count() == 0:
        raise ValueError(f"Menu item {menu_item_id} does not exist.")
    
    # Check for active orders referencing this item
    orders = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {SCHEMA}.seat_order_item
        WHERE menu_item_id = {menu_item_id}
    """).collect()[0]["cnt"]
    if orders > 0:
        raise ValueError(f"Cannot delete menu item {menu_item_id}: it has {orders} active order(s). Mark it unavailable instead.")
    
    item_row = item.collect()[0]
    spark.sql(f"DELETE FROM {SCHEMA}.menu_item WHERE menu_item_id = {menu_item_id}")
    print(f"Deleted menu item '{item_row['name']}' (id={menu_item_id}).")
    return True

In [0]:
# ════════════════════════════════════════════════════════════
# 5. EMPLOYEE CRUD
# ════════════════════════════════════════════════════════════
# create_employee, get_employee, get_all_employees,
# update_employee, delete_employee

def create_employee(restaurant_id, first_name, last_name, role, phone):
    """
    Create a new employee.
    
    Parameters:
        restaurant_id (int): The restaurant this employee works at.
        first_name    (str): Employee first name.
        last_name     (str): Employee last name.
        role          (str): Job role ('manager' or 'server').
        phone         (str): Phone number.
    
    Returns:
        int: The new employee_id.
    """
    # Validate restaurant exists
    restaurant = spark.sql(f"SELECT 1 FROM {SCHEMA}.restaurant WHERE restaurant_id = {restaurant_id}")
    if restaurant.count() == 0:
        raise ValueError(f"Restaurant {restaurant_id} does not exist.")
    
    if role not in ('manager', 'server'):
        raise ValueError("Role must be 'manager' or 'server'.")
    
    spark.sql(f"""
        INSERT INTO {SCHEMA}.employee (restaurant_id, first_name, last_name, role, phone, hire_date, status)
        VALUES ({restaurant_id}, '{first_name}', '{last_name}', '{role}', '{phone}', current_date(), 'active')
    """)
    
    new_emp_id = spark.sql(f"""
        SELECT employee_id FROM {SCHEMA}.employee
        WHERE restaurant_id = {restaurant_id} AND first_name = '{first_name}' AND last_name = '{last_name}'
        ORDER BY employee_id DESC LIMIT 1
    """).collect()[0]["employee_id"]
    
    print(f"Created employee {first_name} {last_name} (id={new_emp_id}, role='{role}').")
    return new_emp_id

def get_employee(employee_id):
    """
    Retrieve a single employee by ID.
    
    Parameters:
        employee_id (int): The employee to look up.
    
    Returns:
        Row: The employee row, or None.
    """
    result = spark.sql(f"""
        SELECT e.employee_id, e.restaurant_id, e.first_name, e.last_name, e.role,
               e.phone, e.hire_date, e.status, r.name AS restaurant_name
        FROM {SCHEMA}.employee e
        JOIN {SCHEMA}.restaurant r ON e.restaurant_id = r.restaurant_id
        WHERE e.employee_id = {employee_id}
    """)
    if result.count() == 0:
        print(f"Employee {employee_id} not found.")
        return None
    row = result.collect()[0]
    print(f"{row['first_name']} {row['last_name']} (id={row['employee_id']}), role={row['role']}, status={row['status']}, restaurant={row['restaurant_name']}.")
    return row

def get_all_employees(restaurant_id, role=None):
    """
    List all employees for a restaurant, optionally filtered by role.
    
    Parameters:
        restaurant_id (int): The restaurant whose employees to list.
        role          (str, optional): Filter by role ('manager' or 'server').
    
    Returns:
        DataFrame: All matching employees.
    """
    filter_clause = f"AND role = '{role}'" if role else ""
    df = spark.sql(f"""
        SELECT employee_id, first_name, last_name, role, phone, hire_date, status
        FROM {SCHEMA}.employee
        WHERE restaurant_id = {restaurant_id} {filter_clause}
        ORDER BY role, last_name, first_name
    """)
    print(f"Employees for restaurant {restaurant_id}{f' (role={role})' if role else ''}:")
    display(df)
    return df

def update_employee(employee_id, first_name=None, last_name=None, role=None, phone=None, status=None):
    """
    Update an employee's fields.
    
    Parameters:
        employee_id (int): The employee to update.
        first_name  (str, optional): New first name.
        last_name   (str, optional): New last name.
        role        (str, optional): New role ('manager' or 'server').
        phone       (str, optional): New phone number.
        status      (str, optional): New status ('active' or 'inactive').
    
    Returns:
        bool: True if the employee was updated.
    """
    emp = spark.sql(f"SELECT * FROM {SCHEMA}.employee WHERE employee_id = {employee_id}")
    if emp.count() == 0:
        raise ValueError(f"Employee {employee_id} does not exist.")
    
    set_clauses = []
    if first_name is not None:
        set_clauses.append(f"first_name = '{first_name}'")
    if last_name is not None:
        set_clauses.append(f"last_name = '{last_name}'")
    if role is not None:
        if role not in ('manager', 'server'):
            raise ValueError("Role must be 'manager' or 'server'.")
        set_clauses.append(f"role = '{role}'")
    if phone is not None:
        set_clauses.append(f"phone = '{phone}'")
    if status is not None:
        if status not in ('active', 'inactive'):
            raise ValueError("Status must be 'active' or 'inactive'.")
        set_clauses.append(f"status = '{status}'")
    
    if not set_clauses:
        print("No fields to update.")
        return False
    
    set_sql = ", ".join(set_clauses)
    spark.sql(f"UPDATE {SCHEMA}.employee SET {set_sql} WHERE employee_id = {employee_id}")
    print(f"Updated employee {employee_id}: {set_sql}.")
    return True

def delete_employee(employee_id):
    """
    Deactivate an employee (soft delete — sets status to 'inactive').
    Fails if the employee has active table assignments.
    
    Parameters:
        employee_id (int): The employee to deactivate.
    
    Returns:
        bool: True if the employee was deactivated.
    """
    emp = spark.sql(f"SELECT * FROM {SCHEMA}.employee WHERE employee_id = {employee_id}")
    if emp.count() == 0:
        raise ValueError(f"Employee {employee_id} does not exist.")
    
    # Check for active table assignments
    assignments = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {SCHEMA}.server_table_assignment
        WHERE employee_id = {employee_id} AND status = 'active'
    """).collect()[0]["cnt"]
    if assignments > 0:
        raise ValueError(f"Cannot deactivate employee {employee_id}: has {assignments} active table assignment(s). Reassign tables first.")
    
    emp_row = emp.collect()[0]
    spark.sql(f"""
        UPDATE {SCHEMA}.employee SET status = 'inactive'
        WHERE employee_id = {employee_id}
    """)
    print(f"Deactivated employee {emp_row['first_name']} {emp_row['last_name']} (id={employee_id}).")
    return True

In [0]:
# ════════════════════════════════════════════════════════════
# 6. SEAT ORDER MANAGEMENT
# ════════════════════════════════════════════════════════════
# assign_food_to_seat, remove_item_from_seat, get_seat_orders,
# update_seat_order, clear_table

def assign_food_to_seat(seat_id, menu_item_id, quantity=1):
    """
    Assign a food item (menu item) to a specific seat at a table.
    
    Parameters:
        seat_id      (int): The seat receiving the order.
        menu_item_id (int): The menu item being ordered.
        quantity     (int): How many of this item (default 1).
    
    Returns:
        int: The seat_order_id.
    """
    # Validate seat exists
    seat = spark.sql(f"SELECT * FROM {SCHEMA}.seat WHERE seat_id = {seat_id}")
    if seat.count() == 0:
        raise ValueError(f"Seat {seat_id} does not exist.")
    
    # Validate menu item exists and is available
    item = spark.sql(f"""
        SELECT * FROM {SCHEMA}.menu_item 
        WHERE menu_item_id = {menu_item_id} AND available = true
    """)
    if item.count() == 0:
        raise ValueError(f"Menu item {menu_item_id} does not exist or is unavailable.")
    
    # Get item name and seat info for the message
    item_row = item.collect()[0]
    seat_row = seat.collect()[0]
    item_name = item_row["name"]
    
    # Insert the order (seat_order_id is GENERATED ALWAYS AS IDENTITY — omit it)
    spark.sql(f"""
        INSERT INTO {SCHEMA}.seat_order_item (seat_id, menu_item_id, quantity, order_time, status)
        VALUES ({seat_id}, {menu_item_id}, {quantity}, current_timestamp(), 'ordered')
    """)
    
    # Retrieve the auto-generated seat_order_id
    new_order_id = spark.sql(f"""
        SELECT seat_order_id FROM {SCHEMA}.seat_order_item 
        WHERE seat_id = {seat_id} AND menu_item_id = {menu_item_id}
        ORDER BY seat_order_id DESC LIMIT 1
    """).collect()[0]["seat_order_id"]
    
    print(f"Assigned {quantity}x {item_name} (menu_item_id={menu_item_id}) to seat {seat_id} (seat_number={seat_row['seat_number']}, table_id={seat_row['table_id']}).")
    return new_order_id

def remove_item_from_seat(seat_order_id):
    """
    Remove a specific food order from a seat.
    
    Parameters:
        seat_order_id (int): The seat_order_id of the order item to remove.
    
    Returns:
        bool: True if the item was removed.
    """
    # Validate the order exists
    order = spark.sql(f"""
        SELECT soi.seat_order_id, soi.seat_id, soi.menu_item_id, soi.quantity, soi.status,
               s.seat_number, s.table_id, mi.name AS item_name
        FROM {SCHEMA}.seat_order_item soi
        JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
        JOIN {SCHEMA}.menu_item mi ON soi.menu_item_id = mi.menu_item_id
        WHERE soi.seat_order_id = {seat_order_id}
    """)
    if order.count() == 0:
        raise ValueError(f"Seat order {seat_order_id} does not exist.")
    
    order_row = order.collect()[0]
    
    # Delete the order item
    spark.sql(f"""
        DELETE FROM {SCHEMA}.seat_order_item
        WHERE seat_order_id = {seat_order_id}
    """)
    
    print(f"Removed {order_row['quantity']}x {order_row['item_name']} (seat_order_id={seat_order_id}) from seat {order_row['seat_number']} at table {order_row['table_id']}.")
    return True

def get_seat_orders(seat_id):
    """
    List all food orders for a specific seat.
    
    Parameters:
        seat_id (int): The seat whose orders to list.
    
    Returns:
        DataFrame: All orders for the seat.
    """
    df = spark.sql(f"""
        SELECT soi.seat_order_id, soi.seat_id, mi.name AS item, mi.category,
               mi.price, soi.quantity, (mi.price * soi.quantity) AS line_total,
               soi.status, soi.order_time
        FROM {SCHEMA}.seat_order_item soi
        JOIN {SCHEMA}.menu_item mi ON soi.menu_item_id = mi.menu_item_id
        WHERE soi.seat_id = {seat_id}
        ORDER BY soi.order_time
    """)
    print(f"Orders for seat {seat_id}:")
    display(df)
    return df

def update_seat_order(seat_order_id, quantity=None, status=None):
    """
    Update the quantity or status of an existing seat order item.
    
    Parameters:
        seat_order_id (int): The order item to update.
        quantity      (int, optional): New quantity (must be > 0).
        status        (str, optional): New status ('ordered', 'served', 'cancelled').
    
    Returns:
        bool: True if the order was updated.
    """
    order = spark.sql(f"SELECT * FROM {SCHEMA}.seat_order_item WHERE seat_order_id = {seat_order_id}")
    if order.count() == 0:
        raise ValueError(f"Seat order {seat_order_id} does not exist.")
    
    set_clauses = []
    if quantity is not None:
        if quantity <= 0:
            raise ValueError("Quantity must be positive. Use remove_item_from_seat to delete.")
        set_clauses.append(f"quantity = {quantity}")
    if status is not None:
        valid_statuses = ['ordered', 'served', 'cancelled']
        if status not in valid_statuses:
            raise ValueError(f"Status must be one of {valid_statuses}.")
        set_clauses.append(f"status = '{status}'")
    
    if not set_clauses:
        print("No fields to update.")
        return False
    
    set_sql = ", ".join(set_clauses)
    spark.sql(f"UPDATE {SCHEMA}.seat_order_item SET {set_sql} WHERE seat_order_id = {seat_order_id}")
    print(f"Updated seat order {seat_order_id}: {set_sql}.")
    return True

def clear_table(table_id):
    """
    Clear a restaurant table: remove all food orders for its seats,
    deactivate the active server assignment, and reset the table to 'available'.
    
    Parameters:
        table_id (int): The table to clear.
    
    Returns:
        bool: True if the table was cleared.
    """
    # Validate table exists
    table = spark.sql(f"SELECT * FROM {SCHEMA}.restaurant_table WHERE table_id = {table_id}")
    if table.count() == 0:
        raise ValueError(f"Table {table_id} does not exist.")
    
    table_row = table.collect()[0]
    table_number = table_row["table_number"]
    
    # 1) Count and delete all seat_order_item records for seats at this table
    order_count = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {SCHEMA}.seat_order_item soi
        JOIN {SCHEMA}.seat s ON soi.seat_id = s.seat_id
        WHERE s.table_id = {table_id}
    """).collect()[0]["cnt"]
    
    if order_count > 0:
        spark.sql(f"""
            DELETE FROM {SCHEMA}.seat_order_item
            WHERE seat_id IN (
                SELECT seat_id FROM {SCHEMA}.seat WHERE table_id = {table_id}
            )
        """)
        print(f"Cleared {order_count} order(s) from table {table_number}.")
    else:
        print(f"No existing orders to clear for table {table_number}.")
    
    # 2) Deactivate the active server assignment (if any)
    active_assignment = spark.sql(f"""
        SELECT assignment_id FROM {SCHEMA}.server_table_assignment 
        WHERE table_id = {table_id} AND status = 'active'
    """)
    if active_assignment.count() > 0:
        spark.sql(f"""
            UPDATE {SCHEMA}.server_table_assignment 
            SET status = 'cleared' 
            WHERE table_id = {table_id} AND status = 'active'
        """)
        print(f"Deactivated server assignment for table {table_number}.")
    else:
        print(f"No active server assignment to deactivate for table {table_number}.")
    
    # 3) Reset the table status to 'available'
    spark.sql(f"""
        UPDATE {SCHEMA}.restaurant_table 
        SET status = 'available' 
        WHERE table_id = {table_id}
    """)
    print(f"Table {table_number} (id={table_id}) is now available.")
    return True

In [0]:
# ── Restaurant CRUD ─────────────────────────────────────────

def create_restaurant(name, address=None, phone=None):
    """Insert a new restaurant and return the generated restaurant_id."""
    spark.sql(f"""
        INSERT INTO {SCHEMA}.restaurant (name, address, phone)
        VALUES ({_q(name)}, {_q(address)}, {_q(phone)})
    """)
    row = spark.sql(f"SELECT max(restaurant_id) AS id FROM {SCHEMA}.restaurant").collect()[0]
    return row["id"]

def get_restaurant(restaurant_id=None):
    """Return all restaurants, or a single restaurant by ID."""
    if restaurant_id is not None:
        return spark.sql(f"SELECT * FROM {SCHEMA}.restaurant WHERE restaurant_id = {restaurant_id}")
    return spark.sql(f"SELECT * FROM {SCHEMA}.restaurant")

def update_restaurant(restaurant_id, **fields):
    """Update one or more columns of a restaurant. Pass column=value kwargs."""
    if not fields:
        raise ValueError("No fields provided to update")
    set_clause = ", ".join(f"{col} = {_q(val)}" for col, val in fields.items())
    spark.sql(f"UPDATE {SCHEMA}.restaurant SET {set_clause} WHERE restaurant_id = {restaurant_id}")

def delete_restaurant(restaurant_id):
    """Delete a restaurant by ID."""
    spark.sql(f"DELETE FROM {SCHEMA}.restaurant WHERE restaurant_id = {restaurant_id}")

# Functions Defined in This Notebook

All **39 functions** are defined locally in this notebook across 7 categories.

## CRUD Functions

| Category | Functions |
| --- | --- |
| **Restaurant Table CRUD** | `create_restaurant_table`, `get_restaurant_table`, `get_all_tables`, `update_restaurant_table`, `delete_restaurant_table` |
| **Server-Table Assignment** | `assign_table_to_server`, `get_table_assignment`, `get_server_tables`, `update_table_assignment`, `remove_server_from_table` |
| **Seat CRUD** | `get_seats`, `add_seat`, `update_seat`, `remove_seat` |
| **Menu Item CRUD** | `create_menu_item`, `get_menu_item`, `get_all_menu_items`, `update_menu_item`, `delete_menu_item` |
| **Employee CRUD** | `create_employee`, `get_employee`, `get_all_employees`, `update_employee`, `delete_employee` |
| **Seat Order Management** | `assign_food_to_seat`, `remove_item_from_seat`, `get_seat_orders`, `update_seat_order`, `clear_table` |
| **Restaurant CRUD** | `create_restaurant`, `get_restaurant`, `update_restaurant`, `delete_restaurant` |

## Imported by

[Restaurant Manager Functions](#notebook-360701697965264) imports these via `%run` and adds **Checkout Operations** (`calculate_table_total`, `calculate_seat_total`, `toggle_discount`, `checkout_together`, `checkout_separate`).